## 1. Configuration

Configure the Portkey client, model, and project paths.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from dotenv import load_dotenv
from portkey_ai import Portkey
import os
import re

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

load_dotenv(ROOT / ".env")

MODEL = os.getenv(
    "PORTKEY_MODEL",
    "@bedrock/qwen.qwen3-coder-next"
)

client = Portkey(
    api_key=os.environ["PORTKEY_API_KEY"],
    base_url=os.getenv(
        "PORTKEY_BASE_URL",
        "https://ai-gateway.apps.cloud.rt.nyu.edu/v1"
    ),
    max_retries=0
)

SOURCE = ROOT / "recovery/recovered_pins.v"
LOG_DIR = ROOT / "ai_logs/readable_rtl"

LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL)
print("Source:", SOURCE)

Model: @bedrock/qwen.qwen3-coder-next
Source: /home/yhh05/projects/aha2026/aha2026-dp1/recovery/recovered_pins.v


## 2. Structural Evidence Extraction

Extract the recovered module interface and ROM structures.

In [3]:
source = SOURCE.read_text()

# Extract the module header.
header = source[:source.index(";") + 1]

# Extract context around ROM primitive instantiations.
rom_sections = []

for match in re.finditer(r"\bSB_RAM40_4K\b", source):
    start = max(0, match.start() - 300)
    end = min(len(source), match.end() + 3500)

    rom_sections.append(source[start:end])

evidence = (
    "MODULE INTERFACE:\n"
    + header
    + "\n\nROM STRUCTURES:\n"
    + "\n\n".join(rom_sections)
)

print("Source characters:", len(source))
print("ROM instances found:", len(rom_sections))
print("Evidence characters:", len(evidence))

Source characters: 217734
ROM instances found: 8
Evidence characters: 30736


## 3. AI Architecture Analysis

Send extracted evidence to the LLM and save the complete interaction.

In [ ]:
SYSTEM_PROMPT = """
You are an expert in FPGA reverse engineering,
SystemVerilog, and cryptographic hardware.

Analyze recovered structural Verilog.

Distinguish confirmed structural facts from hypotheses.
Do not invent an encryption algorithm.
Do not claim functional equivalence without verification.
"""

USER_PROMPT = f"""
We recovered this circuit from a Lattice iCE40 FPGA bitstream.

Our goal is eventually to reconstruct readable,
high-level SystemVerilog with equivalent functionality.

Analyze the supplied structural evidence.

Provide:

1. Confirmed structural observations.
2. Possible ROM functions and datapath organization.
3. What information is missing.
4. What additional netlist regions should be examined.
5. A concrete plan for reconstructing readable RTL.

Do not generate replacement RTL yet.
Do not assume the extracted snippets describe the whole design.

{evidence}
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT}
]

existing = sorted(LOG_DIR.glob("attempt_*"))
attempt = len(existing) + 1

run_dir = LOG_DIR / f"attempt_{attempt:03d}"
run_dir.mkdir()

(run_dir / "prompt.txt").write_text(
    f"Model: {MODEL}\n"
    f"Timestamp: {datetime.now(timezone.utc).isoformat()}\n\n"
    f"[SYSTEM]\n{SYSTEM_PROMPT}\n\n"
    f"[USER]\n{USER_PROMPT}\n"
)

try:
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=4096
    )

    result = response.choices[0].message.content

    (run_dir / "response.txt").write_text(result)
    (run_dir / "response_raw.json").write_text(
        response.model_dump_json(indent=2)
    )

    print(result)

except Exception as exc:
    (run_dir / "error.txt").write_text(repr(exc))
    raise